In [ ]:
import os  # paths: see config/paths.example.env
import anndata as ad
import mudata as md
import scvi
from pathlib import Path
import scanpy as sc
#import muon as mu
import re
import pandas as pd

In [ ]:
adata = ad.read_h5ad(os.environ["HDC_DIR"] + "/MultiVI/Raw/MultiVI_EB_posttraining_adata.h5ad", backed = "r")

In [ ]:
adata.obs.Batch_Lane.value_counts()

In [ ]:
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["ribo"], inplace=True, percent_top=[20], log1p=True
)

In [ ]:
adata

In [ ]:
adata_multiome = ad.read_h5ad(os.environ["HDC_DIR"] + "/MultiVI/HDC_RemoveHighRibo_posttraining_adata.h5ad", backed = "r")

In [ ]:
adata_multiome

In [ ]:
common_barcodes = [x for x in adata.obs_names if x in adata_multiome.obs_names]
len(common_barcodes)

In [ ]:
rna_barcodes = [adata.obs_names[i] for i in range(adata.n_obs) if adata.obs.Batch_Lane.iloc[i] == "EB_RNA"]
len(rna_barcodes)

In [ ]:
all_barcodes = common_barcodes + rna_barcodes
len(all_barcodes)

In [ ]:
adata_filtered = adata[all_barcodes, ].copy(filename = os.environ["HDC_DIR"] + "/MultiVI/Raw/MultiVI_EB_posttraining_adata_filtered.h5ad")
adata_filtered

In [ ]:
MULTIVI_LATENT_KEY = "X_multivi"

#adata.obsm[MULTIVI_LATENT_KEY] = model.get_latent_representation()
#sc.pp.neighbors(adata, use_rep=MULTIVI_LATENT_KEY)
#sc.tl.umap(adata, min_dist=0.2)
# sc.pl.umap(mdata, color=['Clusters', 'Pool'])

for thisResolution in [0.2, 0.5, 1.2, 1.5, 2]:
    print(thisResolution)
    MULTIVI_CLUSTERS_KEY = "leiden_MultiVIResolution_" + str(thisResolution)
    sc.tl.leiden(adata_filtered, resolution = thisResolution, key_added=MULTIVI_CLUSTERS_KEY)
    #sc.tl.dendrogram(adata_filtered, groupby = MULTIVI_CLUSTERS_KEY, use_rep=MULTIVI_LATENT_KEY)
    #sc.pl.dendrogram(adata_filtered, groupby = MULTIVI_CLUSTERS_KEY, save = "HDC_withRNA_RemoveHighRibo_MultiVI_dendrogram_" + MULTIVI_CLUSTERS_KEY + ".png")
    #sc.pl.dendrogram(adata, groupby = MULTIVI_CLUSTERS_KEY, save = "ICC_MultiVI_dendrogram_Resolution_1.pdf")
    sc.pl.umap(adata_filtered, color=[MULTIVI_CLUSTERS_KEY, 'Batch_Lane'], save = "HDC_withRNA_RemoveHighRibo_MultiVI_UMAP_" + MULTIVI_CLUSTERS_KEY + "_Pool.png")

In [ ]:
germlayerlabel = pd.read_csv(os.environ["HDC_DIR"] + "/MultiVI/highres_clusters_labels.tsv.gz", delimiter="\t")
germlayerlabel

In [ ]:
new_barcodes = [re.sub("_lane", "_Lane", x) for x in germlayerlabel["Unnamed: 0"]]
new_barcodes = [re.sub("-1", "_paired", x) for x in new_barcodes]

In [ ]:
germlayerlabel["Barcode"] = new_barcodes
germlayerlabel

In [ ]:
tmp = adata_filtered.obs.copy()
tmp["Barcode"] = tmp.index
tmp = pd.merge(tmp, germlayerlabel, on = "Barcode", how = "left")
tmp.set_index("Barcode", inplace=True)
tmp

In [ ]:
adata_filtered.obs = tmp.copy()
adata_filtered

In [ ]:
adata_filtered.write_h5ad(os.environ["HDC_DIR"] + "/MultiVI/HDC_withRNA_RemoveHighRibo_posttraining_adata.h5ad")

In [ ]:
tmp = adata_multiome.obs.copy()
tmp["Barcode"] = tmp.index
tmp = pd.merge(tmp, germlayerlabel, on = "Barcode", how = "left")
tmp.set_index("Barcode", inplace=True)
tmp

In [ ]:
adata_multiome.obs

In [ ]:
len(list(set(new_barcodes).intersection(adata_multiome.obs_names)))

In [ ]:
adata_multiome.obs = tmp.copy()

In [ ]:
adata_multiome.write_h5ad(os.environ["HDC_DIR"] + "/MultiVI/HDC_multiome_RemoveHighRibo_posttraining_adata_germlayer.h5ad")

In [ ]:
adata_early = ad.read_h5ad(os.environ["HDC_DIR"] + "/MultiVI/Raw/MultiVI_EB_EarlyRNA_posttraining_adata.h5ad", backed = "r")
adata_early.obs

In [ ]:
adata_early.obs.Batch_Lane.value_counts()

In [ ]:
common_barcodes = [x for x in adata_early.obs_names if x in adata_multiome.obs_names]
rna_barcodes = [adata_early.obs_names[i] for i in range(adata_early.n_obs) if adata_early.obs.Batch_Lane.iloc[i] == "Early"]
all_barcodes = common_barcodes + rna_barcodes
print(len(common_barcodes))
print(len(rna_barcodes))
print(len(all_barcodes))



In [ ]:
adata_filtered = adata_early[all_barcodes, ].copy(filename = os.environ["HDC_DIR"] + "/MultiVI/Raw/MultiVI_EB_EarlyRNA_posttraining_adata_filtered.h5ad")

In [ ]:
adata_filtered

In [ ]:
MULTIVI_LATENT_KEY = "X_multivi"

#adata.obsm[MULTIVI_LATENT_KEY] = model.get_latent_representation()
#sc.pp.neighbors(adata, use_rep=MULTIVI_LATENT_KEY)
#sc.tl.umap(adata, min_dist=0.2)
# sc.pl.umap(mdata, color=['Clusters', 'Pool'])

for thisResolution in [0.2, 0.5, 1.2, 1.5, 2]:
    print(thisResolution)
    MULTIVI_CLUSTERS_KEY = "leiden_MultiVIResolution_" + str(thisResolution)
    sc.tl.leiden(adata_filtered, resolution = thisResolution, key_added=MULTIVI_CLUSTERS_KEY)
    #sc.tl.dendrogram(adata_filtered, groupby = MULTIVI_CLUSTERS_KEY, use_rep=MULTIVI_LATENT_KEY)
    #sc.pl.dendrogram(adata_filtered, groupby = MULTIVI_CLUSTERS_KEY, save = "HDC_withRNA_RemoveHighRibo_MultiVI_dendrogram_" + MULTIVI_CLUSTERS_KEY + ".png")
    #sc.pl.dendrogram(adata, groupby = MULTIVI_CLUSTERS_KEY, save = "ICC_MultiVI_dendrogram_Resolution_1.pdf")
    sc.pl.umap(adata_filtered, color=[MULTIVI_CLUSTERS_KEY, 'Batch_Lane'], save = "HDC_EarlyRNA_RemoveHighRibo_MultiVI_UMAP_" + MULTIVI_CLUSTERS_KEY + "_Pool.png")

In [ ]:
tmp = adata_filtered.obs.copy()
tmp["Barcode"] = tmp.index
tmp = pd.merge(tmp, germlayerlabel, on = "Barcode", how = "left")
tmp.set_index("Barcode", inplace=True)
tmp

In [ ]:
adata_filtered.obs

In [ ]:
adata_filtered.obs = tmp.copy()
adata_filtered.write_h5ad(os.environ["HDC_DIR"] + "/MultiVI/HDC_EarlyRNA_RemoveHighRibo_posttraining_adata.h5ad")